# Precios y retornos

En esta sesión examinaremos el comportamiento de algunos precios de activos y sus retornos

### Cargamos las librerias y los datos

Para la programación en R, seguiremos mayormente el flujo de trabajo y la sintaxsis de [Forecasting Principles and Practice](https://otexts.com/fpp3/)

In [ ]:
library(readxl)
library(dplyr)
library(ggpubr)
library(tsibble)
library(feasts)
library(tidyquant)
library(ggplot2)
library(ggtime)
library(moments)


Para ilustrar usaremos dos activos, Bitcoin, y la acción de Ecopetrol en NYSE (ADS). Cargamos los datos de cierre diario usando Yahoo Finance

In [ ]:
btc_close<-tq_get("BTC-USD", get = "stock.prices",from="2022-01-01")|>select(date, close)
btc_close<-as_tsibble(btc_close,index=date)

ec_close<-tq_get("EC", get = "stock.prices",from="2022-01-01")|>select(date, close)

Para cada uno de los activos hacemos la transformación logarítmica y calculamos el retorno diario $r_t=ln(P_t/P_{t-1})$

In [ ]:
btc_close<-btc_close|>mutate(lclose=log(close),dlclose=lclose-lag(lclose))
ec_close<-ec_close|>mutate(lclose=log(close),dlclose=lclose-lag(lclose))

In [ ]:
btc.plot<-ggplot(btc_close,aes(x=date,y=close))+geom_line()+
  labs(title="Precio BTC-USD",x="", y="USD")+theme_minimal()
dbtc.plot<-ggplot(btc_close,aes(x=date,y=dlclose))+geom_line()+
  labs(title="BTC, retorno diario",x="", y="Retorno")+theme_minimal()

ggarrange(btc.plot,dbtc.plot,ncol=1)

In [ ]:
ec.plot<-ggplot(ec_close,aes(x=date,y=close))+geom_line()+
  labs(title="Precio Ecopetrol,ADS",x="", y="USD")+theme_minimal()
dec.plot<-ggplot(ec_close,aes(x=date,y=dlclose))+geom_line()+
  labs(title="EC, retorno diario",x="", y="Retorno")+theme_minimal()

ggarrange(ec.plot,dec.plot,ncol=1)

Ahora examinamos La función de densidad para los retornos y calculamos sus momentos muestrales

In [ ]:
dbtc.density<-ggplot(btc_close,aes(x=dlclose))+geom_density(color="blue",fill="blue",alpha=0.5)+
  theme_minimal()+labs(title="Densidad empírica BTC",x="log return")
dec.density<-ggplot(ec_close,aes(x=dlclose))+geom_density(color="blue",fill="blue",alpha=0.5)+
  theme_minimal()+labs(title="Densidad empírica EC",x="log return")

ggarrange(dbtc.density,dec.density,ncol=1)

In [ ]:
btc_moments<-btc_close|>as_tibble()|>
  summarise(mdlclose=mean(dlclose,na.rm=TRUE),
                                  sddlclose=sd(dlclose,na.rm=TRUE),
                                skewdlclose=skewness(dlclose,na.rm=TRUE),
                              kdlclose=kurtosis(dlclose,na.rm=TRUE)-3)
print(btc_moments)

In [ ]:
ec_moments<-ec_close|>as_tibble()|>
  summarise(mdlclose=mean(dlclose,na.rm=TRUE),
                                  sddlclose=sd(dlclose,na.rm=TRUE),
                                skewdlclose=skewness(dlclose,na.rm=TRUE),
                              kdlclose=kurtosis(dlclose,na.rm=TRUE)-3)
print(ec_moments)

Como se observa en las grtáficas, y se verifica con la estadística descriptiva, los retornos tienen exceso de curtosis, el retorno promedio es cercano a cero, y la asimetría no es mayor problema. 

## Podemos calcular los retornos acumulados y el CAGR

Calculamos los retornos acumulados para todo el periodo

In [ ]:
btc_cum<-sum(btc_close$dlclose,na.rm=TRUE)
print(btc_cum)

Que no es otra cosa que el retorno promedio por el número de días de negociación

In [ ]:
mean(btc_close$dlclose,na.rm=TRUE)*(nrow(btc_close)-1)

El retorno anualizado lo computamos como el acumulado sobre el número de años

In [ ]:
btc_anual<-btc_cum/((nrow(btc_close)-1)/365)
print(btc_anual)

Que al expresarlo como el CAGR es

In [ ]:
btc_cagr<-exp(btc_anual)-1
print(btc_cagr)

### Actividad

- Elija dos series financieras de su interés, consiga los datos en frecuencia diaria, y realice el análisis anterior
- Para todas las series, agregue los datos a frecuencia mensual. Es decir, debe tener el retorno mensual para cada serie, y realice nuevamente los gráficos de densidad y las estadísticas descriptivas. ¿La agregación cambia las propiedades de las series?